# SkyFusion Evaluation From GitHub

Notebook này clone repo nghiên cứu từ GitHub, chạy evaluation cần thiết, copy artifact cần giữ sang `final_artifacts`, rồi xóa repo clone để output Kaggle nhẹ hơn trước khi `Save Version`.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import os
import json

REPO_URL = "https://github.com/Thanhmay2406/tiny-yolo-srw-gcame.git"
REPO_BRANCH = "main"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "tiny-yolo-srw-gcame"
FINAL_DIR = WORK_ROOT / "final_artifacts"
SKYFUSION_DATA = "/kaggle/input/datasets/thanhmay2406/dataset-for-research/SkyFusion_yolo/data.yaml"
BASELINE_RUN = "baseline_yolov8s"

DETECTION_RUNS = [
    "tradaug_yolov8s_seed0",
    "lsal_only_mse_seed0",
    "srw_lsal_p3_mse_seed0",
    "srw_lsal_energy_seed0",
    "srw_lsal_energy_bg_sizeaware_seed0",
]

XAI_RUNS = [
    "lsal_only_mse_seed0",
    "srw_lsal_p3_mse_seed0",
    "srw_lsal_energy_seed0",
    "srw_lsal_energy_bg_sizeaware_seed0",
]

ERROR_ANALYSIS_PAIRS = [
    ("baseline_yolov8s", "srw_lsal_energy_bg_seed0"),
    ("baseline_yolov8s", "srw_lsal_p3_warmup_decay_seed0"),
]

MODEL_SELECTION_RUNS = [
    "baseline_yolov8s",
    "tradaug_yolov8s_seed0",
    "lsal_only_mse_seed0",
    "srw_only_p3_seed0",
    "srw_lsal_p3_mse_seed0",
    "srw_lsal_p3_warmup_decay_seed0",
    "srw_lsal_energy_seed0",
    "srw_lsal_energy_bg_seed0",
    "srw_lsal_energy_bg_sizeaware_seed0",
]

RUN_DETECTION = True
RUN_XAI = True
RUN_ERROR_ANALYSIS = True
RUN_MODEL_SELECTION = True

def run(cmd, cwd=None):
    print("\n[RUN]", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def detection_metrics_path(run_name):
    return REPO_DIR / "experiments" / "skyfusion" / run_name / "detection_eval" / "metrics.json"

def weights_path(run_name):
    return REPO_DIR / "experiments" / "skyfusion" / run_name / "weights" / "best.pt"

def maybe_run_detection_eval(run_name):
    metrics_path = detection_metrics_path(run_name)
    if metrics_path.is_file():
        print(f"[SKIP] detection_eval already exists for {run_name}: {metrics_path}")
        return True
    weights = weights_path(run_name)
    if not weights.is_file():
        print(f"[WARN] Missing weights for {run_name}: {weights}")
        return False
    run([
        "python",
        "scripts/13_eval_detection.py",
        "--data", SKYFUSION_DATA,
        "--weights", str(weights),
        "--split", "valid",
        "--run-name", run_name,
    ], cwd=REPO_DIR)
    return metrics_path.is_file()

def available_detection_runs(run_names):
    return [run_name for run_name in run_names if detection_metrics_path(run_name).is_file()]

FINAL_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_dir": str(REPO_DIR),
    "final_dir": str(FINAL_DIR),
    "skyfusion_data": SKYFUSION_DATA,
}, indent=2))

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run(["python", "-m", "pip", "install", "--upgrade", "pip"], cwd=REPO_DIR)
run(["python", "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)
run(["python", "scripts/00_env_check.py", "--experiments-dir", "experiments"], cwd=REPO_DIR)

## Detection Eval

In [ ]:
if RUN_DETECTION:
    for run_name in [BASELINE_RUN] + DETECTION_RUNS:
        maybe_run_detection_eval(run_name)

## XAI Eval

In [ ]:
if RUN_XAI:
    for run_name in XAI_RUNS:
        weights = REPO_DIR / "experiments" / "skyfusion" / run_name / "weights" / "best.pt"
        if not weights.is_file():
            print(f"[WARN] Missing weights for {run_name}: {weights}")
            continue
        run([
            "python",
            "scripts/14_eval_xai.py",
            "--data", SKYFUSION_DATA,
            "--weights", str(weights),
            "--split", "valid",
            "--target-layers", "P3",
            "--device", "cuda:0",
            "--output-dir", f"experiments/skyfusion/{run_name}/xai_eval",
        ], cwd=REPO_DIR)

## Error Analysis

In [ ]:
if RUN_ERROR_ANALYSIS:
    for baseline_run, candidate_run in ERROR_ANALYSIS_PAIRS:
        baseline_weights = weights_path(baseline_run)
        candidate_weights = weights_path(candidate_run)
        if not baseline_weights.is_file():
            print(f"[WARN] Skip error analysis because baseline weights are missing: {baseline_weights}")
            continue
        if not candidate_weights.is_file():
            print(f"[WARN] Skip error analysis because candidate weights are missing: {candidate_weights}")
            continue
        run([
            "python",
            "scripts/16_error_analysis.py",
            "--dataset-yaml", SKYFUSION_DATA,
            "--baseline-run", baseline_run,
            "--candidate-run", candidate_run,
            "--split", "val",
            "--max-samples", "128",
            "--device", "cuda:0",
            "--output-dir", f"experiments/skyfusion/{candidate_run}/error_analysis_vs_baseline",
        ], cwd=REPO_DIR)

## Model Selection Summary

In [ ]:
if RUN_MODEL_SELECTION:
    available_runs = available_detection_runs(MODEL_SELECTION_RUNS)
    print("[INFO] detection-evaluable runs:", available_runs)
    if BASELINE_RUN not in available_runs:
        print(f"[WARN] Skip model selection because baseline detection_eval is missing for {BASELINE_RUN}")
    elif len(available_runs) < 2:
        print("[WARN] Skip model selection because not enough runs have detection_eval")
    else:
        run([
            "python",
            "scripts/18_generate_model_selection_table.py",
            "--runs",
            *available_runs,
            "--baseline-run",
            BASELINE_RUN,
        ], cwd=REPO_DIR)

## Collect Artifacts And Remove Repo Clone

In [ ]:
def copy_if_exists(src: Path, dst: Path):
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"[COPIED] {src} -> {dst}")
    elif src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"[COPIED] {src} -> {dst}")
    else:
        print(f"[WARN] Missing artifact: {src}")

artifact_runs = sorted(set([BASELINE_RUN] + DETECTION_RUNS + XAI_RUNS + [pair[1] for pair in ERROR_ANALYSIS_PAIRS] + ["srw_only_p3_seed0"]))

for run_name in artifact_runs:
    src_run = REPO_DIR / "experiments" / "skyfusion" / run_name
    dst_run = FINAL_DIR / "experiments" / "skyfusion" / run_name
    for subdir in ["detection_eval", "xai_eval", "error_analysis_vs_baseline", "convergence_eval"]:
        copy_if_exists(src_run / subdir, dst_run / subdir)
    for filename in ["results.csv", "args.yaml", "config.yaml", "hyp.yaml"]:
        copy_if_exists(src_run / filename, dst_run / filename)

for relative_file in [
    "paper/tables/model_selection_summary.md",
    "paper/tables/model_selection_summary.csv",
    "paper/tables/experiment_artifact_inventory.md",
    "paper/tables/experiment_artifact_inventory.csv",
]:
    copy_if_exists(REPO_DIR / relative_file, FINAL_DIR / relative_file)

shutil.rmtree(REPO_DIR)
print(f"[DONE] Removed repo clone: {REPO_DIR}")

In [ ]:
for path in sorted(FINAL_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(FINAL_DIR), path.stat().st_size)